# 📊 Prompt Diff Visualizer
**Series: 2/4** **Author:** Master Timo
**Stack:** 100% Local (Ollama + `nomic-embed-text` + Python standard libraries)

### The Goal
When optimizing a prompt, how do you know if your changes actually altered the core meaning of the output, or if the model just rephrased things? 

This notebook provides an automated framework to run a baseline prompt ($P_1$) and an optimized prompt ($P_2$) across a test dataset, quantifying the differences using two distinct layers:
1. **Lexical/Structural Layer:** Direct line-by-line textual changes using Python's native `difflib`.
2. **Semantic Vector Layer:** Measuring abstract shifts in meaning by calculating the **Cosine Similarity** of local vector embeddings.

### Dependencies
Before running, ensure you have pulled the embedding model locally:
```bash
ollama pull nomic-embed-text

### Code Implementation

In [2]:
import requests
import json
import difflib
import math

In [3]:
OLLAMA_URL = "http://localhost:11434"
GENERATION_MODEL = "mistral"
EMBEDDING_MODEL = "nomic-embed-text"

In [4]:
def run_generation(prompt: str, temperature: float = 0.4) -> str:
    """Generate text using the local LLM."""
    payload = {
        "model": GENERATION_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }
    response = requests.post(f"{OLLAMA_URL}/api/generate", json=payload)
    response.raise_for_status()
    return response.json()["response"].strip()

In [5]:
def get_embedding(text: str) -> list:
    """Generate a vector embedding using a local model via Ollama."""
    payload = {
        "model": EMBEDDING_MODEL,
        "prompt": text
    }
    response = requests.post(f"{OLLAMA_URL}/api/embeddings", json=payload)
    response.raise_for_status()
    return response.json()["embedding"]

In [6]:

def calculate_cosine_similarity(vec_a: list, vec_b: list) -> float:
    """Compute mathematical cosine similarity between two vectors."""
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    magnitude_a = math.sqrt(sum(a * a for a in vec_a))
    magnitude_b = math.sqrt(sum(b * b for b in vec_b))
    if not magnitude_a or not magnitude_b:
        return 0.0
    return dot_product / (magnitude_a * magnitude_b)


In [7]:

def visual_text_diff(text_a: str, text_b: str):
    """Print a clean line-by-line structural diff visualization."""
    diff = difflib.ndiff(text_a.splitlines(), text_b.splitlines())
    print("\n" + "─" * 60 + "\n📜 STRUCTURAL TEXT DIFF\n" + "─" * 60)
    for line in diff:
        if line.startswith('+ ') or line.startswith('- '):
            print(line)
    print("─" * 60)


In [8]:

def evaluate_prompt_diff(input_data: str, baseline_template: str, optimized_template: str):
    """Pipeline to execute prompts, run metrics, and render evaluations."""
    p1 = baseline_template.format(input_data=input_data)
    p2 = optimized_template.format(input_data=input_data)
    
    print("⏳ Running Baseline (P1)...")
    output_1 = run_generation(p1)
    
    print("⏳ Running Optimized (P2)...")
    output_2 = run_generation(p2)
    
    # 1. Structural Comparison
    visual_text_diff(output_1, output_2)
    
    # 2. Semantic Matrix Computation
    vec_1 = get_embedding(output_1)
    vec_2 = get_embedding(output_2)
    similarity_score = calculate_cosine_similarity(vec_1, vec_2)
    semantic_drift = 1.0 - similarity_score
    
    print("\n" + "─" * 60 + "\n🎯 SEMANTIC METRICS\n" + "─" * 60)
    print(f"Vector Alignment (Cosine Similarity): {similarity_score:.4f}")
    print(f"Calculated Semantic Drift (Shift):    {semantic_drift:.4f}")
    print("─" * 60)
    
    if similarity_score > 0.95:
        print("💡 Takeaway: Changes were largely cosmetic/stylistic. Minimal shift in core meaning.")
    elif similarity_score > 0.80:
        print("💡 Takeaway: Moderate semantic shift. The underlying concepts match, but formatting or emphasis changed.")
    else:
        print("💡 Takeaway: High semantic variance. Your optimized prompt altered how the system reasons or outputs.")


In [9]:
# Test Setup
test_input = "Our e-commerce checkout page crashes if a user tries to apply two discount codes concurrently."

p1_template = "Summarize this engineering issue: {input_data}"
p2_template = """Analyze this bug report. Provide a crisp 1-sentence root cause summary, followed by a bulleted list of immediate downstream technical risks. 
Issue: {input_data}"""

evaluate_prompt_diff(test_input, p1_template, p2_template)

⏳ Running Baseline (P1)...
⏳ Running Optimized (P2)...

────────────────────────────────────────────────────────────
📜 STRUCTURAL TEXT DIFF
────────────────────────────────────────────────────────────
- The engineering issue at hand involves the e-commerce checkout page, which experiences a crash when a user attempts to apply two discount codes simultaneously. This problem indicates an incompatibility or flaw in the system's code that handles multiple discount applications, leading to an unanticipated error and the page's failure to function properly. To resolve this issue, developers need to identify the root cause and implement improvements to ensure the checkout page can handle multiple discount codes without crashing.
+ Root Cause Summary: The e-commerce checkout page crash is due to inadequate code handling for applying multiple discount codes simultaneously.
+ 
+ Immediate Downstream Technical Risks:
+ 
+ 1. Potential data loss or corruption during the crash, affecting user order